# SimGuard — Synthetic CDR Dataset Generator

Stochastic time-series simulation of mobile data sessions across 1 000 employees.

**Volume model:** lognormal baseline × weekday/weekend seasonality × AR(1) noise, disaggregated into Poisson-drawn sessions per day.

**Misuse:** two-state Markov chain (Normal ↔ Misuse). Labels live only in `synthetic_cdr_labels.csv` — never in the session table.

## 1· Imports & Config

In [1]:
import os
import glob
import math
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless-safe backend (works in nbconvert too)
import matplotlib.pyplot as plt
import kagglehub

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
rng  = np.random.default_rng(SEED)

N_SIMS        = 1_000
SIM_DAYS      = 90
SIM_START     = pd.Timestamp("2025-01-01")
AVG_SESSIONS_PER_DAY = 5    # Poisson lambda for sessions per SIM per day

OUT_DIR = r"C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data"
os.makedirs(OUT_DIR, exist_ok=True)

SESSION_PATH   = os.path.join(OUT_DIR, "synthetic_cdr_sessions.csv")
LABEL_PATH     = os.path.join(OUT_DIR, "synthetic_cdr_labels.csv")
DAILY_AGG_PATH = os.path.join(OUT_DIR, "synthetic_cdr_daily_agg.csv")

print(f"Seed        : {SEED}")
print(f"SIMs        : {N_SIMS}   Days: {SIM_DAYS}")
print(f"Outputs to  : {OUT_DIR}")
print(f"Expected rows (approx): {N_SIMS * SIM_DAYS * AVG_SESSIONS_PER_DAY:,}")

Seed        : 42
SIMs        : 1000   Days: 90
Outputs to  : C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data
Expected rows (approx): 450,000


C:\Users\ADMIN\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 · Load Employee Dataset

Tries kagglehub first, falls back to local archive.

In [2]:
LOCAL_ARCHIVE = r"C:\Users\ADMIN\Downloads\archive"

dataset_path  = None
_kaggle_error = None

try:
    dataset_path = kagglehub.dataset_download("smayanj/employee-records-dataset")
    print(f"kagglehub: cached at {dataset_path}")
except Exception as exc:
    _kaggle_error = exc
    print(f"kagglehub unavailable ({type(exc).__name__}: {exc})")
    print(f"Falling back to local archive: {LOCAL_ARCHIVE}")
    if not os.path.isdir(LOCAL_ARCHIVE):
        raise FileNotFoundError(
            f"Local archive not found at '{LOCAL_ARCHIVE}'. "
            "Either configure Kaggle credentials or place the dataset there."
        ) from exc
    dataset_path = LOCAL_ARCHIVE

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(
        f"No CSV files under '{dataset_path}'. Contents: {os.listdir(dataset_path)}"
    )
csv_path = max(csv_files, key=os.path.getsize)    # employee_records.csv >> Salary_dataset.csv
print(f"Using CSV : {csv_path}  ({os.path.getsize(csv_path)/1e6:.2f} MB)")

employees_full = pd.read_csv(csv_path)
print(f"Full shape: {employees_full.shape}")
print(f"Columns   : {list(employees_full.columns)}")
employees_full.head(3)

kagglehub: cached at C:\Users\ADMIN\.cache\kagglehub\datasets\smayanj\employee-records-dataset\versions\1
Using CSV : C:\Users\ADMIN\.cache\kagglehub\datasets\smayanj\employee-records-dataset\versions\1\employee_records.csv  (2.01 MB)
Full shape: (30000, 8)
Columns   : ['Employee_ID', 'Employee_Name', 'Age', 'Country', 'Department', 'Position', 'Salary', 'Joining_Date']


,Employee_ID,Employee_Name,Age,Country,Department,Position,Salary,Joining_Date
0,1,Daniel Taylor,25,UK,HR,Analyst,142278.32,2023-06-04
1,2,Ethan Brown,44,India,Marketing,Executive,98549.20,2018-01-13
2,3,Sophia Martinez,51,Japan,Finance,Developer,85565.84,2015-04-30


In [3]:
employees = (
    employees_full
    .sample(n=N_SIMS, random_state=SEED, replace=len(employees_full) < N_SIMS)
    .reset_index(drop=True)
)
print(f"Working sample shape: {employees.shape}")
employees.head(3)

Working sample shape: (1000, 8)


,Employee_ID,Employee_Name,Age,Country,Department,Position,Salary,Joining_Date
0,2309,Mia Thomas,35,Japan,Finance,Analyst,44914.87,2017-11-23
1,22405,Ava Thomas,52,Japan,Marketing,Assistant,50768.15,2019-06-23
2,23398,Sophia Thomas,57,Germany,HR,Developer,69654.52,2016-07-08


## 3 · Cohort Tier Assignment

Keyword match on `Position` → tier → bundle size and usage stats.

In [4]:
TIER_KEYWORDS = {
    "field":   ["sales", "field", "technician", "driver", "delivery"],
    "manager": ["manager", "director", "executive", "head", "chief", "vp"],
    "support": ["assistant", "clerk", "support", "intern", "associate"],
    # "professional" is the default — no keywords needed
}

def assign_tier(position: str) -> str:
    pos_lower = str(position).lower()
    for tier, kws in TIER_KEYWORDS.items():
        if any(kw in pos_lower for kw in kws):
            return tier
    return "professional"

employees["tier"] = employees["Position"].apply(assign_tier)
print("Tier distribution:")
print(employees["tier"].value_counts().to_string())

Tier distribution:
tier
professional    521
manager         329
support         150


## 4 · Tier Parameters

Anchored to CA-Kenya Q3 FY2025/26 average (15.1 GB/month).

| Tier | Bundle | Mean MB/day | CV |
|------|--------|-------------|----|
| field | 100 GB | 1 250 | 0.60 |
| manager | 60 GB | 750 | 0.40 |
| professional | 40 GB | 500 | 0.45 |
| support | 20 GB | 250 | 0.35 |

In [5]:
TIER_PARAMS = {
    "field":        {"bundle_gb": 100, "mean_mb": 1_250, "cv": 0.60},
    "manager":      {"bundle_gb":  60, "mean_mb":   750, "cv": 0.40},
    "professional": {"bundle_gb":  40, "mean_mb":   500, "cv": 0.45},
    "support":      {"bundle_gb":  20, "mean_mb":   250, "cv": 0.35},
}
print("Tier params:", list(TIER_PARAMS.keys()))

Tier params: ['field', 'manager', 'professional', 'support']


## 5 · Per-SIM Lognormal Baseline

$$\\sigma^2 = \\ln(1 + CV^2), \\qquad \\mu = \\ln(\\bar{V}) - \\frac{\\sigma^2}{2}$$

In [6]:
def lognorm_params(mean: float, cv: float):
    """Convert (mean, CV) -> (mu, sigma) for np.random.lognormal."""
    sigma2 = math.log(1 + cv ** 2)          # underlying normal variance from CV squared
    mu     = math.log(mean) - sigma2 / 2    # underlying normal mean adjusted for skewness
    return mu, math.sqrt(sigma2)

baseline_mb = np.empty(N_SIMS)
bundle_gb   = np.empty(N_SIMS)

for idx, row in employees.iterrows():
    p = TIER_PARAMS[row["tier"]]
    mu, sigma        = lognorm_params(p["mean_mb"], p["cv"])
    baseline_mb[idx] = rng.lognormal(mu, sigma)   # personal baseline draw
    bundle_gb[idx]   = p["bundle_gb"]

employees["baseline_mb"] = baseline_mb
employees["bundle_gb"]   = bundle_gb

print("Per-tier baseline_mb summary:")
print(employees.groupby("tier")["baseline_mb"]
      .describe()[["mean","std","min","max"]].round(1))

Per-tier baseline_mb summary:
               mean    std    min     max
tier                                     
manager       737.7  279.3  259.1  2369.7
professional  489.7  217.6   95.2  1593.5
support       252.6   92.2  100.8   570.6


## 6· Daily Volume — AR(1) Noise

$$V_t = V_{\\text{baseline}} \\cdot S(t) \\cdot e^{\\epsilon_t}, \\quad \\epsilon_t = \\phi\\,\\epsilon_{t-1} + \\eta_t$$

$\\phi = 0.4$, $\\sigma_{\\eta} = 0.15$. $S(t) = 0.5$ weekends, $1.0$ weekdays.

In [7]:
AR1_PHI       = 0.40
AR1_SIGMA_ETA = 0.15

dates       = pd.date_range(SIM_START, periods=SIM_DAYS, freq="D")
# Seasonality: weekend = 0.5, weekday = 1.0
seasonality = np.where(dates.dayofweek >= 5, 0.5, 1.0)   # shape (SIM_DAYS,)

def simulate_ar1_volumes(baseline: float, rng_inst) -> np.ndarray:
    """V_t = baseline * S(t) * exp(epsilon_t), AR(1) epsilon."""
    eta     = rng_inst.normal(0, AR1_SIGMA_ETA, size=SIM_DAYS)  # innovations
    epsilon = np.empty(SIM_DAYS)
    epsilon[0] = eta[0]
    for t in range(1, SIM_DAYS):
        epsilon[t] = AR1_PHI * epsilon[t - 1] + eta[t]          # AR(1) persistence: previous shock plus new innovation
    return baseline * seasonality * np.exp(epsilon)              # baseline MB * day-of-week seasonality * AR(1) noise

volume_matrix = np.empty((N_SIMS, SIM_DAYS))
for i in range(N_SIMS):
    volume_matrix[i, :] = simulate_ar1_volumes(employees.loc[i, "baseline_mb"], rng)

print(f"volume_matrix: {volume_matrix.shape}")
print(f"min={volume_matrix.min():.1f}  mean={volume_matrix.mean():.1f}  max={volume_matrix.max():.1f} MB")

volume_matrix: (1000, 90)
min=32.0  mean=464.5  max=3432.4 MB


## 7 · Misuse Injection — Markov Chain

```
Normal  ─ p=0.002 ►  Misuse(category)
Misuse  ─ q(cat)  ►  Normal
```

| Category | q | E[duration] | Volume transform |
|----------|---|-------------|------------------|
| unauthorized_sharing | 1/14 | 14 d | ×2.5 + hotspot_flag=1 |
| data_reselling | 1/10 | 10 d | +Uniform(500,3000) @ 30% |
| ghost_usage | 1/20 | 20 d | fixed 5 MB/day |
| inactive_billing | 1/18 | 18 d | 0 MB; billed = 90% bundle |

In [8]:
MISUSE_CATS      = ["unauthorized_sharing", "data_reselling", "ghost_usage", "inactive_billing"]
CATEGORY_WEIGHTS = np.array([0.35, 0.25, 0.25, 0.15])   # must sum to 1
CATEGORY_Q       = {
    "unauthorized_sharing": 1 / 14,
    "data_reselling":       1 / 10,
    "ghost_usage":          1 / 20,
    "inactive_billing":     1 / 18,
}
P_ENTRY = 0.002   # P(Normal -> Misuse) per day

def apply_volume_transform(base_vol: float, category: str,
                           bundle_gb: float, rng_inst) -> tuple:
    """Return (usage_mb, billed_mb) after applying category transform."""
    ep_len = int(1 / CATEGORY_Q[category])   # expected episode length for billed_mb calc
    if category == "unauthorized_sharing":
        # Simulates hotspot tethering with 2.5x volume inflation
        usage  = base_vol * 2.5              # volume multiplier x2.5
        billed = usage
    elif category == "data_reselling":
        # Simulates data reselling with sporadic large volume bursts
        burst  = rng_inst.uniform(500, 3_000) if rng_inst.random() < 0.30 else 0.0
        usage  = base_vol + burst
        billed = usage
    elif category == "ghost_usage":
        # Simulates zombie SIM with minimal background signaling only
        usage  = 5.0                         # fixed 5 MB background signaling
        billed = usage
    elif category == "inactive_billing":
        # Simulates inactive subscription billed with zero network usage
        usage  = 0.0                         # no real traffic
        billed = (bundle_gb * 1_000 * 0.9) / ep_len  # 90% of bundle billed
    else:
        raise ValueError(f"Unknown category: {category}")
    return usage, billed

usage_matrix  = volume_matrix.copy()
billed_matrix = volume_matrix.copy()
# state_matrix holds 'normal' or category name for each (SIM, day)
state_matrix  = np.full((N_SIMS, SIM_DAYS), "normal", dtype=object)

episode_log = []   # ground-truth: one dict per episode

for sim_idx in range(N_SIMS):
    state    = "normal"
    category = None
    ep_start = None
    b_gb     = employees.loc[sim_idx, "bundle_gb"]

    for day in range(SIM_DAYS):
        base_vol = volume_matrix[sim_idx, day]

        if state == "normal":
            if rng.random() < P_ENTRY:           # entry roll: transition from normal to misuse episode
                category = rng.choice(MISUSE_CATS, p=CATEGORY_WEIGHTS)
                state    = category
                ep_start = day
            usage_matrix[sim_idx, day]  = base_vol
            billed_matrix[sim_idx, day] = base_vol
        else:
            usage, billed = apply_volume_transform(base_vol, category, b_gb, rng)
            usage_matrix[sim_idx, day]  = usage
            billed_matrix[sim_idx, day] = billed
            state_matrix[sim_idx, day]  = category

            if rng.random() < CATEGORY_Q[category]:   # exit roll: transition from misuse back to normal
                episode_log.append({"sim_id": sim_idx, "category": category,
                                    "start_day": ep_start, "end_day": day})
                state = "normal"; category = None; ep_start = None

    if state != "normal":   # close any open episode at window end
        episode_log.append({"sim_id": sim_idx, "category": category,
                            "start_day": ep_start, "end_day": SIM_DAYS - 1})

labels_df = pd.DataFrame(episode_log)
print(f"Episodes logged: {len(labels_df)}")
if not labels_df.empty:
    print(labels_df["category"].value_counts().to_string())

Episodes logged: 176
category
unauthorized_sharing    70
ghost_usage             43
data_reselling          42
inactive_billing        21


## 8· Geo-Velocity

Detection signal only — not a misuse category. Attributed to whatever Markov state is active.

In [9]:
# Real Nairobi-area tower coordinates
TOWER_POOL = [
    # (lat, lon, label)
    (-1.2921, 36.8219, "CBD"),
    (-1.3031, 36.7073, "Westlands"),
    (-1.2195, 36.8870, "Thika Road"),
    (-1.3667, 36.7667, "Karen"),
    (-1.2639, 36.8010, "Upperhill"),
    (-1.2400, 36.9000, "Ruiru"),
    (-1.3172, 36.8272, "Ngong Road"),
    (-1.2833, 36.8167, "Kilimani"),
    (-1.1765, 36.9640, "Juja"),
    (-1.3741, 36.6544, "Ngong"),
]

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in km via the haversine formula."""
    R = 6_371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2 - lat1) / 2) ** 2
         + math.cos(phi1) * math.cos(phi2)
         * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))  # great-circle distance between coordinates in km

N_TOWERS_POOL = len(TOWER_POOL)
VEL_FLAG_KMH  = 900.0          # impossibly fast terrestrial speed
PROB_SAME_TOWER = 0.85          # probability of remaining at current cell tower
PROB_GEO_ANOMALY = 0.005        # probability of triggering impossible-velocity tower jump

# Assign 3 personal towers per SIM (home=0, work=1, other=2)
sim_tower_indices = np.array([
    rng.choice(N_TOWERS_POOL, size=3, replace=False) for _ in range(N_SIMS)
])
print(f"Tower pool: {N_TOWERS_POOL} sites  |  tower matrix: {sim_tower_indices.shape}")

Tower pool: 10 sites  |  tower matrix: (1000, 3)


## 9 · Session Generation (ITU Schema)

Per (SIM, day): Poisson session count → Dirichlet volume split → timestamps → geo-velocity → row emit.

> `call_destination` is always null (reserved). `roaming_flag` is a 0.5% placeholder, not tied to misuse.

In [10]:
def random_dest(rng_inst):
    """Returns None (null) — reserved for future feature work."""
    return None   # intentionally left null; see markdown note above

session_rows  = []
session_id_ctr = 0

for sim_idx in range(N_SIMS):
    b_gb       = employees.loc[sim_idx, "bundle_gb"]
    tower_idxs = sim_tower_indices[sim_idx]           # 3 pool indices for this SIM
    towers     = [TOWER_POOL[i] for i in tower_idxs]  # (lat, lon, label) triples

    prev_tower_local = 0    # start at home tower (local index 0)
    prev_hour        = 0.0  # track previous session's hour for elapsed_h

    for day in range(SIM_DAYS):
        state     = state_matrix[sim_idx, day]    # 'normal' or misuse category
        daily_vol = usage_matrix[sim_idx, day]    # MB already transformed by Markov step

        n_sess = max(1, int(rng.poisson(AVG_SESSIONS_PER_DAY)))

        if n_sess == 1:
            sess_vols = np.array([daily_vol])
        else:
            # Dirichlet(ones) = uniform split with variability
            sess_vols = daily_vol * rng.dirichlet(np.ones(n_sess))  # partition daily volume into random session shares

        if state == "data_reselling":
            # Concentrate burst activity at off-peak hours (20:00-02:00)
            raw_hours = rng.uniform(20, 26, n_sess)   # 26 wraps to 02:00
            hours = np.sort(raw_hours % 24)
        else:
            # Business hours: 07:00-22:00
            hours = np.sort(rng.uniform(7, 22, n_sess))

        date_base = dates[day]   # pd.Timestamp for this day

        for sess_i in range(n_sess):
            is_geo_anomaly = bool(rng.random() < PROB_GEO_ANOMALY)

            if is_geo_anomaly:  # anomaly branch: implausible speed via tiny time gap
                # Switch to a different tower with implausibly short time gap
                other_locals    = [i for i in range(3) if i != prev_tower_local]
                cur_tower_local = int(rng.choice(other_locals))
                elapsed_h       = float(rng.uniform(0.01, 0.20))   # 0.6-12 min gap
            else:  # normal mobility branch: home-biased tower selection
                if rng.random() < PROB_SAME_TOWER:
                    cur_tower_local = prev_tower_local
                else:
                    others          = [i for i in range(3) if i != prev_tower_local]
                    cur_tower_local = int(rng.choice(others))
                hour_gap  = max(hours[sess_i] - prev_hour, 0.01)
                elapsed_h = hour_gap

            lat1, lon1, _  = towers[prev_tower_local]
            lat2, lon2, lbl = towers[cur_tower_local]
            dist_km        = haversine_km(lat1, lon1, lat2, lon2)
            velocity_kmh   = dist_km / elapsed_h   # haversine dist / elapsed time
            # geo_flag: detection signal, NOT a misuse category
            geo_velocity_flag = int(velocity_kmh > VEL_FLAG_KMH)

            dl_frac     = float(rng.beta(7, 3))    # mean ~0.70 download fraction
            sess_vol    = float(sess_vols[sess_i])
            download_mb = round(sess_vol * dl_frac, 4)
            upload_mb   = round(sess_vol * (1.0 - dl_frac), 4)

            apn = "public.internet" if rng.random() < 0.08 else "corporate.apn"

            hotspot_flag = 1 if state == "unauthorized_sharing" else 0  # flag active tethering during unauthorized sharing

            session_duration = max(10, int(rng.lognormal(math.log(300), 0.5)))

            session_ts = date_base + pd.Timedelta(hours=float(hours[sess_i]))

            roaming_flag = 1 if rng.random() < 0.005 else 0

            cell_tower_id = f"NBI-{tower_idxs[cur_tower_local]:03d}"

            session_rows.append({
                # Core identifiers
                "session_id":        session_id_ctr,
                "sim_id":            sim_idx,
                "session_timestamp": session_ts,
                "session_duration":  session_duration,
                # Traffic metrics
                "event_type":        "Data",   # Voice/SMS reserved for future notebooks
                "download_mb":       download_mb,
                "upload_mb":         upload_mb,
                "apn":               apn,
                "hotspot_flag":      hotspot_flag,
                # Spatial telemetry
                "cell_tower_id":     cell_tower_id,
                "latitude":          lat2,
                "longitude":         lon2,
                "geo_velocity_flag": geo_velocity_flag,   # detection signal only
                # Destination & network context (reserved / placeholder)
                "call_destination":  random_dest(rng),
                "roaming_flag":      roaming_flag,
            })

            session_id_ctr  += 1
            prev_tower_local = cur_tower_local
            prev_hour        = hours[sess_i]

print(f"Sessions generated: {len(session_rows):,}")
sessions_df = pd.DataFrame(session_rows)

sessions_df["sim_id"]           = sessions_df["sim_id"].astype(np.int32)
sessions_df["session_duration"] = sessions_df["session_duration"].astype(np.int32)
sessions_df["hotspot_flag"]     = sessions_df["hotspot_flag"].astype(np.int8)
sessions_df["roaming_flag"]     = sessions_df["roaming_flag"].astype(np.int8)
sessions_df["geo_velocity_flag"]= sessions_df["geo_velocity_flag"].astype(np.int8)
sessions_df["download_mb"]      = sessions_df["download_mb"].astype(np.float32)
sessions_df["upload_mb"]        = sessions_df["upload_mb"].astype(np.float32)

print(f"sessions_df shape: {sessions_df.shape}")
print(f"Columns: {list(sessions_df.columns)}")
sessions_df.head(3)

Sessions generated: 449,877
sessions_df shape: (449877, 15)
Columns: ['session_id', 'sim_id', 'session_timestamp', 'session_duration', 'event_type', 'download_mb', 'upload_mb', 'apn', 'hotspot_flag', 'cell_tower_id', 'latitude', 'longitude', 'geo_velocity_flag', 'call_destination', 'roaming_flag']


,session_id,sim_id,session_timestamp,session_duration,event_type,download_mb,upload_mb,apn,hotspot_flag,cell_tower_id,latitude,longitude,geo_velocity_flag,call_destination,roaming_flag
0,0,0,2025-01-01 08:12:14.361727494,572,Data,123.351700,96.932602,corporate.apn,0,NBI-008,-1.1765,36.9640,0,None,0
1,1,0,2025-01-01 10:29:51.167858801,298,Data,45.837799,16.043301,public.internet,0,NBI-009,-1.3741,36.6544,0,None,0
2,2,0,2025-01-01 11:03:27.442447124,184,Data,28.165001,22.910500,corporate.apn,0,NBI-009,-1.3741,36.6544,0,None,0


## 10· Ground Truth Labels

Written to a separate CSV — never joined into the session table.

In [11]:
if not labels_df.empty:
    labels_df["duration_days"] = labels_df["end_day"] - labels_df["start_day"] + 1
    labels_df = labels_df[["sim_id", "category", "start_day", "end_day", "duration_days"]]

print(f"Labels shape: {labels_df.shape}")
print()
if not labels_df.empty:
    print("Episode duration stats per category:")
    print(labels_df.groupby("category")["duration_days"]
          .describe()[["count","mean","std","min","max"]].round(1).to_string())
labels_df.head(5)

Labels shape: (176, 5)

Episode duration stats per category:
                      count  mean   std  min   max
category                                          
data_reselling         42.0   9.4   7.5  1.0  30.0
ghost_usage            43.0  15.7  12.8  1.0  48.0
inactive_billing       21.0  12.0  11.5  3.0  53.0
unauthorized_sharing   70.0  11.7  11.9  1.0  52.0


,sim_id,category,start_day,end_day,duration_days
0,1,unauthorized_sharing,73,89,17
1,19,data_reselling,64,82,19
2,22,ghost_usage,24,57,34
3,28,unauthorized_sharing,31,36,6
4,31,unauthorized_sharing,3,6,4


## 11· Voice & SMS Generation

Voice calls and SMS are **monitored telemetry only** — no misuse injection, no Markov state,
no anomaly labels. They populate the ITU schema `event_type` field and give a second resource
axis (airtime / SMS allowance) separate from the data bundle.

**Probabilistic model (Erlang / ITU-T E.501 telephony traffic theory):**
- Daily call count ~ Poisson(λ): canonical model for call-arrival rates in teletraffic engineering.
- Call duration ~ Exponential(mean = 102.9 s): standard queueing-theory holding-time distribution;
  memoryless property makes it tractable and fits empirical mobile call-duration distributions for bulk simulation.
- Daily SMS count ~ Poisson(λ): same rationale as calls.

**Sourced parameters — CA Kenya Q2 FY2025/26 Sector Statistics Report:**
- Blended mean call duration: 1.71 min (102.9 s), weighted on-net 83.1% @ 1.8 min + off-net 16.9% @ 1.3 min.
- National λ_calls = 2.61 calls/day (133.9 MoU min/month ÷ 1.71 min/call ÷ 30 days).
- National λ_sms = 2.04 SMS/day (61.1 SMS/month ÷ 30 days).

**Tier scaling — inherited assumption, not independently sourced:**
No published role-based voice/SMS breakdown exists. The same multipliers used for data bundles
(field 2.5×, manager 1.5×, professional 1× anchor, support 0.5×) are applied here as a
modelling convenience only.

| Tier | λ_calls/day | λ_sms/day | Voice allowance (min/mo) | SMS allowance/mo |
|------|-------------|-----------|--------------------------|------------------|
| field | 6.53 | 5.10 | 300 | 225 |
| manager | 3.92 | 3.06 | 180 | 135 |
| professional | 2.61 | 2.04 | 120 | 90 |
| support | 1.31 | 1.02 | 60 | 45 |

In [12]:
MEAN_CALL_DURATION_S = 102.9  # seconds, blended on-net/off-net (CA Kenya Q2 FY2025/26)

# Allowances set at ~1.5× expected monthly usage:
#   expected_min/mo = lambda_calls × 1.71 min × 30 days
#   'field' tier is defined for completeness but no employees currently match it.
TIER_VOICE_SMS_PARAMS = {
    "field":        {"lambda_calls": 6.53, "lambda_sms": 5.10,
                     "voice_allowance_min": 500, "sms_allowance": 340},
    "manager":      {"lambda_calls": 3.92, "lambda_sms": 3.06,
                     "voice_allowance_min": 300, "sms_allowance": 205},
    "professional": {"lambda_calls": 2.61, "lambda_sms": 2.04,
                     "voice_allowance_min": 200, "sms_allowance": 135},
    "support":      {"lambda_calls": 1.31, "lambda_sms": 1.02,
                     "voice_allowance_min": 100, "sms_allowance":  70},
}
print("Voice/SMS tier params:", list(TIER_VOICE_SMS_PARAMS.keys()))

Voice/SMS tier params: ['field', 'manager', 'professional', 'support']


In [13]:
voice_sms_rows   = []
voice_session_id = int(sessions_df["session_id"].max()) + 1  # continue from data sessions

# Tower probability: home-biased, mirrors data session behaviour
_TOWER_PROBS = np.array([0.70, 0.20, 0.10])

for sim_idx in range(N_SIMS):
    tier       = employees.loc[sim_idx, "tier"]
    vp         = TIER_VOICE_SMS_PARAMS[tier]
    tower_idxs = sim_tower_indices[sim_idx]  # 3 personal towers (same as data)
    towers     = [TOWER_POOL[i] for i in tower_idxs]

    for day in range(SIM_DAYS):
        date_base = dates[day]

        # Voice calls
        n_calls = int(rng.poisson(vp["lambda_calls"]))  # daily calls drawn from tier Poisson rate
        for _ in range(n_calls):
            dur_s       = max(1, int(rng.exponential(MEAN_CALL_DURATION_S)))  # call duration from exponential distribution
            t_local     = int(rng.choice(3, p=_TOWER_PROBS))
            lat, lon, _ = towers[t_local]
            tower_id    = f"NBI-{tower_idxs[t_local]:03d}"
            hour        = float(rng.uniform(7, 22))
            ts          = date_base + pd.Timedelta(hours=hour)
            roaming     = 1 if rng.random() < 0.005 else 0
            dest        = f"+254{int(rng.integers(700_000_000, 800_000_000))}"
            voice_sms_rows.append({
                "session_id":        voice_session_id,
                "sim_id":            sim_idx,
                "session_timestamp": ts,
                "session_duration":  dur_s,
                "event_type":        "Voice",
                "download_mb":       0.0,
                "upload_mb":         0.0,
                "apn":               None,
                "hotspot_flag":      0,
                "cell_tower_id":     tower_id,
                "latitude":          lat,
                "longitude":         lon,
                "geo_velocity_flag": 0,
                "call_destination":  dest,
                "roaming_flag":      roaming,
            })
            voice_session_id += 1

        # SMS
        n_sms = int(rng.poisson(vp["lambda_sms"]))  # daily SMS drawn from tier Poisson rate
        for _ in range(n_sms):
            t_local     = int(rng.choice(3, p=_TOWER_PROBS))
            lat, lon, _ = towers[t_local]
            tower_id    = f"NBI-{tower_idxs[t_local]:03d}"
            hour        = float(rng.uniform(7, 22))
            ts          = date_base + pd.Timedelta(hours=hour)
            roaming     = 1 if rng.random() < 0.005 else 0
            dest        = f"+254{int(rng.integers(700_000_000, 800_000_000))}"
            voice_sms_rows.append({
                "session_id":        voice_session_id,
                "sim_id":            sim_idx,
                "session_timestamp": ts,
                "session_duration":  0,
                "event_type":        "SMS",
                "download_mb":       0.0,
                "upload_mb":         0.0,
                "apn":               None,
                "hotspot_flag":      0,
                "cell_tower_id":     tower_id,
                "latitude":          lat,
                "longitude":         lon,
                "geo_velocity_flag": 0,
                "call_destination":  dest,
                "roaming_flag":      roaming,
            })
            voice_session_id += 1

voice_sms_df = pd.DataFrame(voice_sms_rows)
voice_sms_df["sim_id"]            = voice_sms_df["sim_id"].astype(np.int32)
voice_sms_df["session_duration"]  = voice_sms_df["session_duration"].astype(np.int32)
voice_sms_df["hotspot_flag"]      = voice_sms_df["hotspot_flag"].astype(np.int8)
voice_sms_df["roaming_flag"]      = voice_sms_df["roaming_flag"].astype(np.int8)
voice_sms_df["geo_velocity_flag"] = voice_sms_df["geo_velocity_flag"].astype(np.int8)
voice_sms_df["download_mb"]       = voice_sms_df["download_mb"].astype(np.float32)
voice_sms_df["upload_mb"]         = voice_sms_df["upload_mb"].astype(np.float32)

# Merge into sessions_df (data + voice + SMS in one table)
sessions_df = pd.concat([sessions_df, voice_sms_df], ignore_index=True)
sessions_df = sessions_df.sort_values(["sim_id", "session_timestamp"]).reset_index(drop=True)

print(f"Voice/SMS rows added: {len(voice_sms_df):,}")
print(f"Total sessions after merge: {len(sessions_df):,}")
print("\nEvent type breakdown:")
print(sessions_df["event_type"].value_counts().to_string())

Voice/SMS rows added: 456,793
Total sessions after merge: 906,670

Event type breakdown:
event_type
Data     449877
Voice    256557
SMS      200236


## 12 · Daily Aggregate Table

Pre-computes day-level features for Isolation Forest and Prophet.

In [14]:
sessions_df["date"] = sessions_df["session_timestamp"].dt.normalize()

daily_agg = (
    sessions_df
    .groupby(["sim_id", "date"])
    .agg(
        total_download_mb  = ("download_mb",       "sum"),
        total_upload_mb    = ("upload_mb",          "sum"),
        session_count      = ("session_id",         "count"),
        total_duration_s   = ("session_duration",   "sum"),
        hotspot_flag_count = ("hotspot_flag",        "sum"),
        geo_flag_count     = ("geo_velocity_flag",   "sum"),
        roaming_sessions   = ("roaming_flag",        "sum"),
        unique_towers      = ("cell_tower_id",      "nunique"),
        apn_public_count   = ("apn",
                              lambda x: (x == "public.internet").sum()),
    )
    .reset_index()
)

daily_agg["total_mb"] = daily_agg["total_download_mb"] + daily_agg["total_upload_mb"]

# Attach tier for convenience
tier_map = employees["tier"].to_dict()
daily_agg["tier"] = daily_agg["sim_id"].map(tier_map)

print(f"daily_agg shape: {daily_agg.shape}")
daily_agg.head(3)

daily_agg shape: (90000, 13)


,sim_id,date,total_download_mb,total_upload_mb,session_count,total_duration_s,hotspot_flag_count,geo_flag_count,roaming_sessions,unique_towers,apn_public_count,total_mb,tier
0,0,2025-01-01,294.560608,220.542206,14,1961,0,0,0,3,1,515.102783,professional
1,0,2025-01-02,306.129822,158.067108,12,1790,0,0,0,3,0,464.196930,professional
2,0,2025-01-03,290.872406,175.930801,8,1819,0,0,0,3,1,466.803223,professional


In [15]:
# Compute per-SIM per-day voice minutes and SMS counts, then merge into daily_agg
voice_daily = (
    sessions_df[sessions_df["event_type"] == "Voice"]
    .groupby(["sim_id", "date"])
    .agg(voice_minutes_used=("session_duration", lambda x: x.sum() / 60.0))
    .reset_index()
)
sms_daily = (
    sessions_df[sessions_df["event_type"] == "SMS"]
    .groupby(["sim_id", "date"])
    .agg(sms_count=("session_id", "count"))
    .reset_index()
)

daily_agg = daily_agg.merge(voice_daily, on=["sim_id", "date"], how="left")
daily_agg = daily_agg.merge(sms_daily,   on=["sim_id", "date"], how="left")
daily_agg["voice_minutes_used"] = daily_agg["voice_minutes_used"].fillna(0.0).round(3)
daily_agg["sms_count"]          = daily_agg["sms_count"].fillna(0).astype(int)

# Monthly cumulative usage and remaining allowance (mirrors data bundle_gb tracking)
daily_agg = daily_agg.sort_values(["sim_id", "date"]).reset_index(drop=True)
daily_agg["_month"] = pd.to_datetime(daily_agg["date"]).dt.to_period("M")

# Month-to-date cumulative voice minutes, mirroring data tracking
daily_agg["_voice_cum"] = (
    daily_agg.groupby(["sim_id", "_month"])["voice_minutes_used"].cumsum()
)
# Month-to-date cumulative SMS count, mirroring data tracking
daily_agg["_sms_cum"] = (
    daily_agg.groupby(["sim_id", "_month"])["sms_count"].cumsum()
)

voice_allow_map = {
    i: TIER_VOICE_SMS_PARAMS[employees.loc[i, "tier"]]["voice_allowance_min"]
    for i in range(N_SIMS)
}
sms_allow_map = {
    i: TIER_VOICE_SMS_PARAMS[employees.loc[i, "tier"]]["sms_allowance"]
    for i in range(N_SIMS)
}

# Remaining voice allowance, mirroring data bundle tracking
daily_agg["voice_minutes_remaining"] = (
    daily_agg["sim_id"].map(voice_allow_map) - daily_agg["_voice_cum"]
).round(2)
# Remaining SMS allowance, mirroring data bundle tracking
daily_agg["sms_remaining"] = (
    daily_agg["sim_id"].map(sms_allow_map) - daily_agg["_sms_cum"]
).astype(int)

daily_agg.drop(columns=["_month", "_voice_cum", "_sms_cum"], inplace=True)

print(f"daily_agg shape after voice/SMS columns: {daily_agg.shape}")
print("Added: voice_minutes_used, sms_count, voice_minutes_remaining, sms_remaining")
daily_agg[["sim_id", "date", "voice_minutes_used", "sms_count",
           "voice_minutes_remaining", "sms_remaining"]].head(5)

daily_agg shape after voice/SMS columns: (90000, 17)
Added: voice_minutes_used, sms_count, voice_minutes_remaining, sms_remaining


,sim_id,date,voice_minutes_used,sms_count,voice_minutes_remaining,sms_remaining
0,0,2025-01-01,3.983,5,196.02,130
1,0,2025-01-02,8.767,1,187.25,129
2,0,2025-01-03,0.000,3,187.25,126
3,0,2025-01-04,1.200,1,186.05,125
4,0,2025-01-05,2.717,3,183.33,122


## 13 · Sanity Checks

In [16]:
print("=== Row counts ===")
print(f"  Raw sessions   : {len(sessions_df):>10,}")
print(f"  Labels         : {len(labels_df):>10,}")
print(f"  Daily agg rows : {len(daily_agg):>10,}")
print()

print("=== Per-tier mean total_mb/day (ALL days, normal + misuse) ===")
print(daily_agg.groupby("tier")["total_mb"].mean().round(1).sort_values(ascending=False).to_string())
print()

print("=== Misuse day counts per category (state_matrix) ===")
anomaly_day_counts = {}
for cat in MISUSE_CATS:
    anomaly_day_counts[cat] = int((state_matrix == cat).sum())
for cat, cnt in anomaly_day_counts.items():
    pct = 100 * cnt / (N_SIMS * SIM_DAYS)
    print(f"  {cat:<25s}: {cnt:>6,} days  ({pct:.2f}%)")
total_anom = sum(anomaly_day_counts.values())
print(f"  {'TOTAL':<25s}: {total_anom:>6,} days  ({100*total_anom/(N_SIMS*SIM_DAYS):.2f}%)")
print()

print("=== hotspot_flag audit ===")
# Add temporary state column for audit ONLY — not saved to CSV
sessions_df["_day"] = sessions_df["session_timestamp"].dt.date
hot_in  = sessions_df[sessions_df["hotspot_flag"] == 1].shape[0]
hot_out = sessions_df[sessions_df["hotspot_flag"] == 0].shape[0]
print(f"  hotspot_flag=1 sessions: {hot_in:,}")
print(f"  hotspot_flag=0 sessions: {hot_out:,}")

# Cross-check: build day-level unauthorized_sharing mask from state_matrix
us_sim_days = set()
for s in range(N_SIMS):
    for d in range(SIM_DAYS):
        if state_matrix[s, d] == "unauthorized_sharing":
            us_sim_days.add((s, d))

sessions_df["_dnum"] = (sessions_df["session_timestamp"].dt.normalize() - SIM_START).dt.days
flagged_in_episode  = sessions_df[
    sessions_df.apply(lambda r: (int(r["sim_id"]), int(r["_dnum"])) in us_sim_days, axis=1)
]["hotspot_flag"].sum()
flagged_outside      = sessions_df["hotspot_flag"].sum() - flagged_in_episode
print(f"  hotspot=1 rows inside  unauthorized_sharing episodes: {flagged_in_episode:,}")
print(f"  hotspot=1 rows outside unauthorized_sharing episodes: {flagged_outside:,}  (should be 0)")
sessions_df.drop(columns=["_day", "_dnum"], inplace=True)
print()

print("=== Geo-velocity flag ===")
geo_tot = sessions_df["geo_velocity_flag"].sum()
print(f"  Total geo-flagged sessions: {geo_tot:,}  ({100*geo_tot/len(sessions_df):.2f}%)")
print()

print("=== Schema guard (no label leakage) ===")
forbidden = {"is_anomalous", "ground_truth_category", "state", "category", "misuse_label"}
found = forbidden & set(sessions_df.columns)
if found:
    print(f"  FAIL — forbidden columns present: {found}")
else:
    print("  PASS — no ground-truth columns in sessions_df")

=== Row counts ===
  Raw sessions   :    906,670
  Labels         :        176
  Daily agg rows :     90,000

=== Per-tier mean total_mb/day (ALL days, normal + misuse) ===
tier
manager         640.200012
professional    428.500000
support         220.100006

=== Misuse day counts per category (state_matrix) ===
  unauthorized_sharing     :    750 days  (0.83%)
  data_reselling           :    354 days  (0.39%)
  ghost_usage              :    634 days  (0.70%)
  inactive_billing         :    230 days  (0.26%)
  TOTAL                    :  1,968 days  (2.19%)

=== hotspot_flag audit ===
  hotspot_flag=1 sessions: 3,637
  hotspot_flag=0 sessions: 903,033
  hotspot=1 rows inside  unauthorized_sharing episodes: 3,637
  hotspot=1 rows outside unauthorized_sharing episodes: 0  (should be 0)

=== Geo-velocity flag ===
  Total geo-flagged sessions: 10,930  (1.21%)

=== Schema guard (no label leakage) ===
  PASS — no ground-truth columns in sessions_df


In [17]:
print("=== Voice/SMS event counts ===")
event_counts = sessions_df["event_type"].value_counts()
print(event_counts.to_string())
n_calls = int(event_counts.get("Voice", 0))
n_sms   = int(event_counts.get("SMS",   0))
print(f"\nTotal calls : {n_calls:,}")
print(f"Total SMS   : {n_sms:,}")

print("\n=== Mean calls/day and SMS/day per tier (observed vs expected) ===")
_v = sessions_df[sessions_df["event_type"] == "Voice"].copy()
_s = sessions_df[sessions_df["event_type"] == "SMS"].copy()
_v["tier"] = _v["sim_id"].map(employees["tier"])
_s["tier"] = _s["sim_id"].map(employees["tier"])
_v["_date"] = _v["session_timestamp"].dt.normalize()
_s["_date"] = _s["session_timestamp"].dt.normalize()
calls_pd = _v.groupby(["tier", "sim_id", "_date"]).size().groupby("tier").mean()
sms_pd   = _s.groupby(["tier", "sim_id", "_date"]).size().groupby("tier").mean()
print("Calls/day:")
for t in sorted(TIER_VOICE_SMS_PARAMS):
    exp = TIER_VOICE_SMS_PARAMS[t]["lambda_calls"]
    obs = calls_pd.get(t, float("nan"))
    print(f"  {t:<15s}: observed={obs:.2f}  expected={exp:.2f}")
print("SMS/day:")
for t in sorted(TIER_VOICE_SMS_PARAMS):
    exp = TIER_VOICE_SMS_PARAMS[t]["lambda_sms"]
    obs = sms_pd.get(t, float("nan"))
    print(f"  {t:<15s}: observed={obs:.2f}  expected={exp:.2f}")

print("\n=== SIMs exceeding monthly allowance (informational only) ===")
sims_over_voice = int((daily_agg.groupby("sim_id")["voice_minutes_remaining"].min() < 0).sum())
sims_over_sms   = int((daily_agg.groupby("sim_id")["sms_remaining"].min() < 0).sum())
print(f"  Voice allowance exceeded in any month : {sims_over_voice} SIM(s)")
print(f"  SMS allowance exceeded in any month   : {sims_over_sms} SIM(s)")

=== Voice/SMS event counts ===
event_type
Data     449877
Voice    256557
SMS      200236

Total calls : 256,557
Total SMS   : 200,236

=== Mean calls/day and SMS/day per tier (observed vs expected) ===
Calls/day:
  field          : observed=nan  expected=6.53
  manager        : observed=4.00  expected=3.92
  professional   : observed=2.83  expected=2.61
  support        : observed=1.80  expected=1.31
SMS/day:
  field          : observed=nan  expected=5.10
  manager        : observed=3.21  expected=3.06
  professional   : observed=2.35  expected=2.04
  support        : observed=1.60  expected=1.02

=== SIMs exceeding monthly allowance (informational only) ===
  Voice allowance exceeded in any month : 25 SIM(s)
  SMS allowance exceeded in any month   : 0 SIM(s)


In [18]:
# Event-type count bar chart
fig, ax = plt.subplots(figsize=(7, 4))
ec      = sessions_df["event_type"].value_counts()
colors  = ["#4C72B0", "#55A868", "#C44E52"]
ax.bar(ec.index, ec.values, color=colors[:len(ec)], edgecolor="white")
ax.set_title("Session counts by event type", fontsize=13, fontweight="bold")
ax.set_ylabel("Sessions")
ax.set_xlabel("Event type")
for i, (et, v) in enumerate(ec.items()):
    ax.text(i, v + max(ec.values) * 0.01, f"{v:,}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
event_plot_path = os.path.join(OUT_DIR, "event_type_counts.png")
plt.savefig(event_plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {event_plot_path}")

Saved: C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\event_type_counts.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30440\3336798380.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tier_means = (
    daily_agg.groupby("tier")["total_mb"]
    .mean()
    .sort_values(ascending=False)
)
colors_t = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
axes[0].bar(tier_means.index, tier_means.values,
            color=colors_t[:len(tier_means)], edgecolor="white")
axes[0].set_title("Mean Daily Total MB by Tier", fontsize=13, fontweight="bold")
axes[0].set_ylabel("MB/day")
axes[0].set_xlabel("Cohort tier")
for i, (t, v) in enumerate(tier_means.items()):
    axes[0].text(i, v + 5, f"{v:.0f}", ha="center", va="bottom", fontsize=10)

cats   = list(anomaly_day_counts.keys())
counts = list(anomaly_day_counts.values())
colors_c = ["#DD8452", "#4878D0", "#6ACC65", "#D65F5F"]
axes[1].bar(cats, counts, color=colors_c, edgecolor="white")
axes[1].set_title("Injected Misuse Days by Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("# SIM-days in misuse state")
axes[1].tick_params(axis="x", rotation=18)
for i, (c, v) in enumerate(zip(cats, counts)):
    axes[1].text(i, v + 2, str(v), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plot1_path = os.path.join(OUT_DIR, "sanity_check_plots.png")
plt.savefig(plot1_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {plot1_path}")

Saved: C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\sanity_check_plots.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30440\1536892700.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
sample_sims = rng.integers(0, N_SIMS, size=4)
fig, axes   = plt.subplots(2, 2, figsize=(14, 7), sharex=True)

for ax, s in zip(axes.ravel(), sample_sims):
    s = int(s)
    sim_daily = daily_agg[daily_agg["sim_id"] == s].sort_values("date")
    ax.plot(sim_daily["date"], sim_daily["total_mb"],
            lw=1, color="steelblue", label="total_mb")

    ymax_val = float(sim_daily["total_mb"].max()) if len(sim_daily) > 0 else 1.0
    for cat in MISUSE_CATS:
        # shade misuse episodes using state_matrix
        cat_days = [d for d in range(SIM_DAYS) if state_matrix[s, d] == cat]
        if cat_days:
            cat_dates = [dates[d] for d in cat_days]
            ax.vlines(cat_dates, 0, ymax_val, alpha=0.25, lw=4,
                      label=cat, colors={"unauthorized_sharing":"#DD8452",
                                         "data_reselling":"#4878D0",
                                         "ghost_usage":"#6ACC65",
                                         "inactive_billing":"#D65F5F"}[cat])
    tier = employees.loc[s, "tier"]
    ax.set_title(f"SIM {s} ({tier})", fontsize=11)
    ax.set_ylabel("MB/day")
    ax.legend(fontsize=6, loc="upper right")

fig.suptitle("Daily Volume — 4 Random SIMs (coloured bars = misuse state)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plot2_path = os.path.join(OUT_DIR, "sample_timeseries.png")
plt.savefig(plot2_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {plot2_path}")

Saved: C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\sample_timeseries.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30440\2783908788.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14 · Save Outputs

| File | Contents |
|------|----------|
| `synthetic_cdr_sessions.csv` | Full ITU session table |
| `synthetic_cdr_labels.csv` | Ground-truth episodes |
| `synthetic_cdr_daily_agg.csv` | Day-level aggregates |

In [21]:
# Drop the temporary 'date' column (session_timestamp is the canonical field)
sessions_save = sessions_df.drop(columns=["date"], errors="ignore")
sessions_save.to_csv(SESSION_PATH, index=False)
print(f"Sessions  -> {SESSION_PATH}  ({len(sessions_save):,} rows)")

if not labels_df.empty:
    labels_df.to_csv(LABEL_PATH, index=False)
    print(f"Labels    -> {LABEL_PATH}  ({len(labels_df):,} episodes)")
else:
    print("WARNING: no misuse episodes logged — labels CSV will be empty.")

daily_agg.to_csv(DAILY_AGG_PATH, index=False)
print(f"Daily agg -> {DAILY_AGG_PATH}  ({len(daily_agg):,} rows)")
print()
print("All outputs saved.")

Sessions  -> C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\synthetic_cdr_sessions.csv  (906,670 rows)
Labels    -> C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\synthetic_cdr_labels.csv  (176 episodes)
Daily agg -> C:\Users\ADMIN\Desktop\projects\simguard\ml-service\data\synthetic_cdr_daily_agg.csv  (90,000 rows)

All outputs saved.
